In [1]:
%load_ext autoreload
%autoreload 2
# %cd /mnt/sdd1/akanksha/formulacode/datasmith
%cd /mnt/sdd1/atharvas/formulacode/datasmith/

import datetime

import pandas as pd

from datasmith.logging_config import get_logger
from src.datasmith.execution.filter_commits import is_delinquient_repo

logger = get_logger("notebooks.building_reports_pr")

curr_date: str = datetime.datetime.now().isoformat()

/mnt/sdd1/atharvas/formulacode/datasmith


In [2]:
commits_df = pd.read_parquet(
    "/mnt/sdd1/atharvas/formulacode/datasmith/scratch/artifacts/pipeflush/merge_commits_filtered_with_patch.parquet"
)

In [12]:
# "1bf0f43e6a43fd2500424ccf6f867370565f6af5" in commits_df["sha"].values
perfonly_df = pd.read_parquet("scratch/artifacts/pipeflush/perfonly_commits_with_patch_sm.parquet")

print(perfonly_df.query("sha == '1bf0f43e6a43fd2500424ccf6f867370565f6af5'").iloc[0]["problem_statement"])

There is still a regression when the BQM has quadratic interactions


In [3]:
print(commits_df.shape)
commits_df["is_delinquient_repo"] = commits_df["repo_name"].apply(is_delinquient_repo)
print(commits_df.query("~is_delinquient_repo").shape)
commits_df.groupby("repo_name").agg(
    num_prs=("pr_number", "nunique"),
    pr_stargazers=("pr_base_stargazers_count", "max"),
    pr_forks=("pr_base_forks_count", "max"),
    last_merge_date=("pr_merged_at", "max"),
).sort_values("num_prs", ascending=False)

(26815, 145)
(26815, 146)


,num_prs,pr_stargazers,pr_forks,last_merge_date
repo_name,,,,
pandas-dev/pandas,3212,46786,19104,2025-10-09T15:39:20Z
scikit-learn/scikit-learn,2452,63625,26313,2025-10-10T14:20:05Z
xdslproject/xdsl,2104,428,125,2025-10-10T09:09:33Z
apache/arrow,2061,16046,3874,2025-08-15T06:41:28Z
scipy/scipy,1450,14075,5494,2025-10-10T08:43:03Z
...,...,...,...,...
spotify/voyager,5,1507,74,2024-12-19T07:44:26Z
scikit-learn-contrib/metric-learn,4,1425,229,2019-03-14T10:19:57Z
jkjkil4/JAnim,3,188,14,2025-08-29T02:16:11Z


In [4]:
small_df = commits_df.query("pr_base_stargazers_count > 2000").sample(25)
for _, row in small_df.iterrows():
    print(row["pr_url"])

https://api.github.com/repos/scikit-learn/scikit-learn/pulls/25815
https://api.github.com/repos/dedupeio/dedupe/pulls/1061
https://api.github.com/repos/pandas-dev/pandas/pulls/47605
https://api.github.com/repos/apache/arrow/pulls/46615
https://api.github.com/repos/scikit-learn/scikit-learn/pulls/22457
https://api.github.com/repos/napari/napari/pulls/4392
https://api.github.com/repos/apache/arrow/pulls/43486
https://api.github.com/repos/apache/arrow/pulls/46723
https://api.github.com/repos/apache/arrow/pulls/43853
https://api.github.com/repos/pandas-dev/pandas/pulls/54685
https://api.github.com/repos/pandas-dev/pandas/pulls/46901
https://api.github.com/repos/pandas-dev/pandas/pulls/52150
https://api.github.com/repos/pandas-dev/pandas/pulls/50011
https://api.github.com/repos/Qiskit/qiskit/pulls/14604
https://api.github.com/repos/scikit-image/scikit-image/pulls/6855
https://api.github.com/repos/scipy/scipy/pulls/22803
https://api.github.com/repos/pandas-dev/pandas/pulls/52330
https://api.

In [9]:
# Updated to use new ReportBuilder interface
from datasmith.scrape.report_builder import ReportBuilder

# sample_pr = "https://api.github.com/repos/dedupeio/dedupe/pulls/997"
# sample_pr = "https://api.github.com/repos/pandas-dev/pandas/pulls/54745"

rb = ReportBuilder(
    enable_llm_backends=True,
    summarize_llm=True,
    add_classification=True,
    filter_performance_only=True,
    max_links_to_follow=60,
    # model_name="@togetherai/meta-llama/Llama-3.3-70B-Instruct-Turbo",
    # model_name="@google/gemini-1.5-flash-latest",
    # model_name="@google/gemini-2.0-flash-exp",
    model_name="local/meta-llama/Llama-3.3-70B-Instruct",
)

results = small_df.apply(rb.build, axis=1)

01:19:34 INFO     openai._base_client: Retrying request to /chat/completions in 0.456676 seconds
01:19:34 INFO     openai._base_client: Retrying request to /chat/completions in 0.956059 seconds
01:19:35 INFO     openai._base_client: Retrying request to /chat/completions in 1.540257 seconds
01:19:40 INFO     openai._base_client: Retrying request to /chat/completions in 0.401344 seconds
01:19:40 INFO     openai._base_client: Retrying request to /chat/completions in 0.923347 seconds
01:19:41 INFO     openai._base_client: Retrying request to /chat/completions in 1.546407 seconds
2025/10/21 01:19:46 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
01:19:46 INFO     openai._base_client: Retrying request to /chat/completions in 0.468975 seconds
01:19:47 INFO     openai._base_client: Retrying request to /chat/completions in 0.987333 seconds
01:19:47 INFO     openai._base_client: Retrying request to /chat/completions in 1.822573 seconds


KeyboardInterrupt: 

In [7]:
results_df = pd.json_normalize(results.apply(lambda d: d.__dict__))
small_df2 = pd.concat([small_df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1)

In [11]:
small_df2.is_performance_commit.value_counts()
# small_df2.iloc[:, -25:]

is_performance_commit
False    21
True      4
Name: count, dtype: int64

In [12]:
for row in small_df2.query("is_performance_commit").to_dict(orient="records"):
    print(f"PR URL: {row['pr_url']}")
    print(f"Classification: {row['classification']}")
    print("-----")
    print("REPORT")
    # print(row["report_md"])
    print("-----")
    print("PROBLEM STATEMENT")
    print(row["problem_statement"])
    print("-----")
    print("HINTS")
    print(row["hints"])
    print("\n\n")

PR URL: https://api.github.com/repos/scikit-learn/scikit-learn/pulls/25815
Classification: The category of this optimization technique is "Remove or reduce work (requirements & UX)" because it reduces the amount of work done by the code by avoiding unnecessary parameter validation.
-----
REPORT
-----
PROBLEM STATEMENT
It happens that estimators or functions call other public estimators or functions, sometimes in a loop, for which parameter validation is done each time. Param validation is cheap but definitely not free and this is something we want to avoid. In addition, once the first validation is done we make sure that internally we pass the appropriate parameters to inner estimators or functions, so these nested validation are useless. For instance, `MiniBatchDictionaryLearning -> _minibatch_step -> _sparse_encode -> _sparse_encode_precomputed -> Lasso`. MinibatchDictionaryLearning calls public functions that call public classes themselves, resulting in validating the parameters and

In [ ]:
from tqdm.auto import tqdm

from datasmith.scrape.build_pr_report import build_pr_report
from datasmith.scrape.report_builder import ReportBuilder

reports = []

# Initialize ReportBuilder once for all reports
rb = ReportBuilder(
    enable_llm_backends=False,
    summarize_llm=False,
    add_classification=True,  # Enable classification for batch processing
    max_links_to_follow=60,
)

# Process reports in batch
h_map = {}
for _, pr in tqdm(small_df.iterrows()):
    if not pr["pr_url"]:
        reports.append(None)
        continue

    print(pr["pr_url"])

    # Convert row to dict and build report
    pr_dict = pr.to_dict()
    result = rb.build(pr_dict=pr_dict)

    # Store results
    h_map[pr["pr_url"]] = (
        result.report_md,
        result.problem_statement,
        result.hints,
        result.classification,
        result.difficulty,
        result.is_performance_commit,
    )
    reports.append(result.report_md)
    break
report = h_map[next(iter(h_map))][0] if h_map else ""

In [ ]:
print(report)


### Hints



### Problem Statement


[ISSUE_NUM] 
I implement the `atcoder.math.pow_mod` as a wrapper of `pow` without introducing additional assertions.
Do we need to modify `docs/math.rst` either?



### Classification

13. Use a higher-level system that optimizes for you


### Difficulty

easy


In [ ]:
report, prob_statement, hints, classification, difficulty, performance_issue = build_pr_report(
    link="https://api.github.com/repos/xarray-contrib/xbatcher/pulls/112",
    summarize_llm=True,
    add_classification=False,
    patch="",
)
print(prob_statement)

## Problem
Generating batches in `__init__` is slow and memory intensive, related to [ISSUE_NUM] and [ISSUE_NUM]. Initialization is changed to load indices into memory rather than corresponding datasets. The change enabled the initialization of a 1.7 TB dataset with 1m+ samples, 30+ spatial features, in about 10 seconds which was previously overloading memory.

Rather than filling `_batches` with `DataArrays` and `Datasets`, it is filled with indices from `selector = {key: slice for key, slice in zip(dims, slices)}`. This required an update to the `concat_input_dims` option where the operation is done in `__getitem__()`. It is possible that this change decreases performance when this `concat_input_dims=True`.

## Related Issues
The issue is related to the following issues:
- Issue 0: Generate batches lazily
- Issue 1: Cache batches

### Issue 0: Generate batches lazily
This issue discusses the need to generate batches lazily to improve performance. A similar implementation was made in 

In [ ]:
from pprint import pprint

pprint(h_map)

{'https://api.github.com/repos/not522/ac-library-python/pulls/55': ('NOT_A_VALID_PR',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    False),
 'https://api.github.com/repos/not522/ac-library-python/pulls/56': ('NOT_A_VALID_PR',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    False),
 'https://api.github.com/repos/not522/ac-library-python/pulls/58': ('NOT_A_VALID_PR',
              

In [ ]:
h_map["https://api.github.com/repos/not522/ac-library-python/pulls/70"]

('\n### Hints\n\n\nThe code has been reviewed and appears to be good to go, with no issues or concerns raised.\n\n\n### Performance Issue\n\n{"label": "YES", "reason": "Implementing pow_mod as a wrapper of pow", "confidence": 80, "flags": ["mentions-speed", "startup"]}\n\n\n### LLM Generated summary\n\n### Problem\nThe issue is about implementing the `atcoder.math.pow_mod` function as a wrapper of the built-in `pow` function without introducing additional assertions.\n\n### Background\nThe discussion started with a comment on Issue 67, which suggested adding `pow_mod` in `math.py`. It was noted that Python\'s built-in `pow` function has a `mod` argument, and it was proposed to introduce `atcoder.math.pow_mod` as a wrapper for consistency with the original ACL.\n\n### Proposed Implementation\nThe user wants to implement the wrapper and has two questions:\n1. Do we need additional assertions in the wrapper?\n2. Do we still need to add tests for that?\n\n### Analysis\nThe original impleme

In [ ]:
count = sum(1 for v in h_map.values() if v[0] not in ("NOT_A_PERFORMANCE_COMMIT", "NOT_A_VALID_PR"))
print(count)

1


In [ ]:
# df = pd.DataFrame([{**c, **{"report": report}} for c, report in zip(commits, reports)])

In [ ]:
# df[["url", "report"]][~df["report"].str.contains("NOT_A_VALID_PR")].url.values

In [ ]:
# logger.info(f"PR Report:\n{report}")
print(report)

NOT_A_VALID_PR
